---

Configure the installation of the postgres database

---

In [1]:
# autoload
%load_ext autoreload
%autoreload 2
from pgl import pglStorage

In [3]:
# configure the installation
configuration = pglStorage.validateConfiguration(postgresPort=5433, installDirectory="~/data/pgl")

Postgres configuration
postgresVersion              17
postgresHost                 127.0.0.1
postgresPort                 5433
databaseName                 pgl
databaseSchema               pgl
databaseUser                 pgl_app
adminUser                    pgl_admin
installDirectory             /Users/justin/data/pgl
dataDirectory                /Users/justin/data/pgl/data
logFile                      /Users/justin/data/pgl/postgres.log
installPostgresIfMissing     True
resetApplicationPassword     False
startupTimeoutSeconds        60
connectTimeoutSeconds        10
brew                         /opt/homebrew/bin/brew
formula                      postgresql@17


---

Install PostgreSQL

---

This installs PostgreSQL binaries if they are missing.

Homebrew may also initialize its own default database directory. This notebook
does not use that directory and does not start the Homebrew-managed service.

Our database will use the dedicated directory configured above. 
Note that if postgres is already installed, then this will not change anything

In [4]:
pglStorage.installPostgres(configuration)

PostgreSQL available: postgres (PostgreSQL) 17.11 (Homebrew)
Executable directory: /opt/homebrew/opt/postgresql@17/bin


---

Initialize the dedicated database directory

---

On the first run, choose an administrator password and save it in your password
manager.

On later runs, enter the existing administrator password.

This cell does not reset administrator passwords or replace existing database
files. A different Postgres major version requires an explicit upgrade.

In [6]:
pglStorage.initPostgres(configuration)

Using existing PostgreSQL cluster: /Users/justin/data/pgl/data


---

Start Postgres

---

Postgres will listen only on `127.0.0.1`. Unix-domain sockets are disabled for
this dedicated instance.

If another server is already using the configured port, this cell stops rather
than silently connecting to it. Choose another port, such as 5433, in the
configuration cell and rerun the relevant cells.

If this instance is already running, it is left running. Changing its port
requires stopping it before starting it again.

In [8]:
pglStorage.startPostgres(configuration)

waiting for server to start.... done
server started
Postgres started at 127.0.0.1:5433.
Server log: /Users/justin/data/pgl/postgres.log


---

Verify postgress login and instance

---

In [10]:
pglStorage.verifyPostgres(configuration)

+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
(pglStorage:_askPassword) Enter password
+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
Verified PostgreSQL 17.11 (Homebrew)
Verified data directory: /Users/justin/data/pgl/data
Verified local-only networking.


---

Create the application account and database

---


Python/pgl uses a non-superuser account, not the administrator account.

On the first run:
- Choose an application password.
- The notebook creates the application role and database.
- The application owns its dedicated schema and can create tables there.

On later runs:
- Enter the existing application password.
- Existing passwords are preserved unless `RESET_APPLICATION_PASSWORD = True`.
- Existing data is not deleted.

The password is needed again when connecting from a fresh Python process.
Save it in your password manager.

In [11]:
pglStorage.createDatabase(configuration)

+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
(pglStorage:_askPassword) Enter password
+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
(pglStorage:_askPassword) Enter password
+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
Application account ready: pgl_app
Database ready: pgl
Schema ready: pgl (owner: pgl_admin)
Application login verified.


---

Test the application connection

---


This verifies that Python can connect using the application account.

It also tests permission to create and write a table in the application schema.
The test runs inside a transaction that is rolled back, leaving no test table
or data behind.

In [13]:
pglStorage.verifyDatabase(configuration)

+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
(pglStorage:_askPassword) Enter password
+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=
Database: pgl
User: pgl_app
Schema: pgl
Application connection, schema selection, and query test passed.
Table read/write testing is deferred until schema migrations have run.
